# 📋 Dive.ai 팀 의사결정 문서
## 문화콘텐츠 데이터 미디어 유형 활용 방향

> **작성 배경**
> 문화콘텐츠 데이터는 영화 / 시리즈 / 소설 / 만화로 나뉘어져 있고, 미디어 유형마다 서사 특성이 달라요.
> 유저가 단순히 소재만 입력할 때 **어떤 유형의 데이터를 참고할 것인지** 결정이 필요합니다.

---

## 1. 왜 미디어 유형 구분이 중요한가?

유저가 "배신당한 복수극" 이렇게만 입력하면,
현재 설계는 `genre` 필터로 검색하는데 **장르와 미디어 유형은 별개**입니다.

예를 들어 "스릴러" 안에는 영화도, 소설도, 만화도 있는데
이걸 섞어서 RAG로 주입하면 **미디어별 서사 특성이 뭉개집니다.**

### 미디어 유형별 서사 특성 비교

| 유형 | 서사 특성 | 분기형 인터랙티브와의 적합도 |
|------|-----------|------------------------------|
| 🎬 **영화** | 씬 전환 빠름, 시각적 묘사 중심, 2시간 내 완결 구조 | 보통 (단선 서사 중심) |
| 📺 **시리즈** | 에피소드별 클리프행어, 떡밥 회수 구조, 분기 선택에 친화적 | **높음** ✅ |
| 📖 **소설** | 내면 묘사 깊음, 긴 호흡, 복잡한 복선 구조 | 보통 (텍스트 밀도 높음) |
| 🎨 **만화** | 컷 단위 임팩트, 대사 간결, 시각적 연출 강조 | 보통 (시각 중심) |

> Dive.ai의 핵심 구조인 **분기형 인터랙티브**와 가장 잘 맞는 건 **시리즈** 데이터입니다.
> 에피소드마다 선택지가 존재하고 엔딩이 분기되는 구조가 시리즈 서사 패턴과 동일하기 때문입니다.

---
## 2. 두 가지 방향

### 🅐 방향 1 — 유저가 미디어 유형을 직접 선택

소재 입력 후 "어떤 형식으로 만들고 싶으세요?" 선택지를 제공합니다.

```
[소재 입력] 배신당한 복수극
[형식 선택] 영화식 / 시리즈식 / 소설식 / 만화식
     ↓
선택한 유형의 데이터만 RAG 참조
```

#### ✅ 장점
- 유저가 원하는 **서사 스타일**을 직접 선택할 수 있어 맞춤형 결과 제공
- 미디어 유형별 특성이 살아있는 시나리오 생성 가능
- 데이터셋 4개 유형을 **골고루 활용** 가능
- 유저에게 선택의 재미와 다양성 제공

#### ❌ 단점
- 유저 입력 단계가 하나 더 추가 → UX 마찰 증가
- 유저가 미디어 유형 차이를 이해 못할 경우 선택 혼란
- 4개 유형의 데이터 양 차이로 결과 편차 발생 가능 (영화 1,431개 vs 소설 203개)

#### 💡 고려사항
- "영화식 / 소설식" 대신 **"빠른 전개 / 깊은 묘사 / 만화 연출"** 같이 유저 친화적 언어로 바꾸면 혼란 줄일 수 있음
- 소설(203개), 만화(425개)는 데이터 수가 적어 RAG 품질이 낮을 수 있음

### 🅑 방향 2 — 시리즈 데이터를 주(主)로 고정

Dive.ai의 분기형 인터랙티브 구조에 가장 잘 맞는 **시리즈 데이터를 기본**으로 사용합니다.
유저는 미디어 유형을 선택하지 않아도 됩니다.

```
[소재 입력] 배신당한 복수극
     ↓
시리즈 데이터 중심으로 RAG 참조
(필요 시 영화·소설·만화 데이터 보조 참조)
```

#### ✅ 장점
- 유저 입력 단계 단순 → **UX 마찰 최소화**
- 시리즈(1,502개)는 4개 유형 중 **가장 많은 데이터** → RAG 품질 안정적
- 에피소드 구조 = 분기형 구조와 일치 → **서비스 컨셉과 가장 잘 맞음**
- 개발 복잡도 낮음 (유형별 분기 로직 불필요)

#### ❌ 단점
- 유저가 "만화 느낌의 시나리오"를 원해도 시리즈 패턴이 주입됨
- 영화·소설·만화 데이터 **활용도가 낮아짐** (데이터셋 투자 대비 효율 감소)
- 서사 스타일 다양성이 줄어들 수 있음

#### 💡 고려사항
- 시리즈를 주로 쓰되, `genre` 필터로 장르 내 시리즈 데이터만 좁혀 쓰면 품질 유지 가능
- 향후 유저 피드백 보고 미디어 유형 선택 기능을 **추후 추가**하는 로드맵도 가능

---
## 3. 방향 비교 요약

| 항목 | 🅐 유저 선택 | 🅑 시리즈 고정 |
|------|-------------|----------------|
| UX 복잡도 | 높음 (선택 단계 추가) | 낮음 (입력 → 바로 생성) |
| 서비스 컨셉 부합도 | 보통 | **높음** (분기형 = 시리즈 구조) |
| 데이터 활용 다양성 | **높음** (4개 유형 모두 활용) | 낮음 (시리즈 중심) |
| RAG 품질 안정성 | 유형마다 편차 있음 | **안정적** (1,502개) |
| 개발 복잡도 | 높음 | **낮음** |
| 유저 맞춤화 | **높음** | 낮음 |
| MVP 적합도 | 보통 | **높음** |

---
## 4. 논의 포인트


1. **서비스 초기(MVP) 단계에서 UX 단순함 vs 유저 맞춤화, 어느 쪽이 더 중요한가?**
2. **소설·만화 데이터(203개, 425개)가 RAG에 쓰기에 충분한 양인가?**
3. **유저가 "미디어 유형"을 선택하는 경험이 Dive.ai의 창작 몰입감에 도움이 되는가, 방해가 되는가?**
4. **방향 🅑로 시작하고 추후 🅐로 확장하는 로드맵이 현실적인가?**